## Day-Ahead Price Scenarios (DK2)

### Data Processing

Historical hourly day-ahead (DA) prices from Nord Pool for the DK2 bidding zone are used to construct realistic price scenarios.

The dataset requires preprocessing:

- The decimal separator is converted from comma to dot to enable numerical operations.
- All values are cast to floating-point format.
- Negative prices are adjusted to avoid unrealistic incentives in the optimization:

$$
\lambda^{DA}_{t} = \max(\lambda^{DA}_{t}, 1)
$$

This ensures numerical stability and prevents distorted bidding behavior.

---

### Scenario Construction

To generate representative price scenarios:

1. The hourly prices are organized into daily profiles (24 hours per day).
2. The average daily price is computed for each day:
$$
\bar{\lambda}_d = \frac{1}{24} \sum_{t=1}^{24} \lambda^{DA}_{t,d}
$$

3. All days are sorted based on $\bar{\lambda}_d$.
4. The dataset is divided into 20 bins.
5. From each bin, the **median day** is selected.

This approach ensures:
- coverage of low, medium, and high price regimes,
- preservation of realistic intra-day price patterns,
- a balanced representation of market conditions.

---

### Output

The final result is a matrix of day-ahead price scenarios:

- Dimensions: $24 \times 20$
- Rows: hours of the day
- Columns: selected scenarios (each corresponding to a historical day)

These price scenarios are later combined with wind and imbalance scenarios to form the full stochastic scenario set.

In [6]:
# ============================================
# STEP 2 — DA PRICE SCENARIOS (DK2)
# ============================================

import pandas as pd
import numpy as np

# ----------------------------
# LOAD EXCEL
# ----------------------------
df_prices = pd.read_excel("raw/Nordpool_prices_manual.xlsx")

# First column = hours → rename
df_prices = df_prices.rename(columns={df_prices.columns[0]: "Hour"})

# ----------------------------
# FIX DECIMAL FORMAT
# ----------------------------
# Replace comma with dot and convert to float

for col in df_prices.columns[1:]:
    df_prices[col] = (
        df_prices[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# ----------------------------
# REMOVE NEGATIVE PRICES
# ----------------------------
df_prices.iloc[:, 1:] = df_prices.iloc[:, 1:].clip(lower=1)

# ----------------------------
# CONVERT TO SCENARIO MATRIX
# ----------------------------
# Shape: (24, N_days)

price_matrix = df_prices.iloc[:, 1:].values

# ----------------------------
# SELECT 20 PRICE SCENARIOS
# ----------------------------
# Same logic as wind → median bins

daily_avg_price = price_matrix.mean(axis=0)

sorted_idx = np.argsort(daily_avg_price)

bins = np.array_split(sorted_idx, 20)

selected_idx = [b[len(b)//2] for b in bins]

price_scenarios = price_matrix[:, selected_idx]

# ----------------------------
# CREATE TABLE
# ----------------------------
scenario_names_price = [
    f"S{i+1} ({df_prices.columns[1:][selected_idx[i]]})"
    for i in range(20)
]

price_df = pd.DataFrame(
    price_scenarios,
    columns=scenario_names_price
)

price_df.insert(0, "Hour", np.arange(24))

print(price_df.head())

   Hour  S1 (2026-03-25 00:00:00)  S2 (2026-03-01 00:00:00)  \
0     0                      1.81                     48.16   
1     1                      1.00                     50.51   
2     2                      1.00                     51.49   
3     3                      1.00                     50.74   
4     4                      1.00                     49.55   

   S3 (2026-02-21 00:00:00)  S4 (2026-04-12 00:00:00)  \
0                     52.44                     35.13   
1                     46.64                     36.06   
2                     43.46                     37.87   
3                     41.65                     41.36   
4                     38.97                     41.21   

   S5 (2026-02-28 00:00:00)  S6 (2026-02-26 00:00:00)  \
0                     80.02                     76.42   
1                     76.30                     72.38   
2                     74.73                     68.73   
3                     71.77                     67

In [7]:
# ============================================
# SAVE ALL TABLES TO data/processed/
# ============================================

import os

# Ensure folder exists
os.makedirs("processed", exist_ok=True)

# ----------------------------
# SAVE PRICES
# ----------------------------
price_df.to_csv(
    "processed/da_price_scenarios.csv",
    index=False
)

print("All files saved in data/processed/")

All files saved in data/processed/
